In [0]:
STORAGE_ACCOUNT = 'stccasemauricioes'
CONTAINER = 'data'
STORAGE_KEY = 'COLOCAR_CHAVE_AQUI'

spark.conf.set("fs.azure.account.key." + STORAGE_ACCOUNT + ".blob.core.windows.net", STORAGE_KEY)

BASE = "wasbs://" + CONTAINER + "@" + STORAGE_ACCOUNT + ".blob.core.windows.net"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Carrega locations
df_loc = spark.read.option('header', True).option('inferSchema', True).csv(f'{BASE}/locations.csv')

# FILTRO CRITICO: tira agregados regionais (World, Europe, EU, "High income")
df_loc_paises = df_loc.filter(~F.col('iso_code').startswith('OWID_'))

# Q2.1 - paises com mais vacinas
print('Q2.1 - Paises que usam mais tipos de vacinas:')
q21 = (df_loc_paises
    .select('location', 'iso_code',
            F.split(F.col('vaccines'), ',\\s*').alias('vaccines_arr'))
    .withColumn('n_vaccines', F.size('vaccines_arr'))
    .orderBy(F.desc('n_vaccines'), 'location'))

q21.show(15, truncate=False)

Q2.1 - Paises que usam mais tipos de vacinas:
+-----------+--------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+
|location   |iso_code|vaccines_arr                                                                                                                                                        |n_vaccines|
+-----------+--------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+
|Iran       |IRN     |[COVIran Barekat, CanSino, Covaxin, Johnson&Johnson, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing, Sinovac, Soberana02, Sputnik Light, Sputnik V]|12        |
|Afghanistan|AFG     |[CanSino, Covaxin, Johnson&Johnson, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing, Sinovac, Sputnik Light, Sputnik V] 

In [0]:
# Q2.2 - top 10 mensal
print('Carregando vaccinations.json...')
df_vac_raw = spark.read.option('multiline', True).json(f'{BASE}/vaccinations.json')

# A estrutura tem 'iso_code', 'country', 'data' (array)
# Explode pra ter uma linha por dia
df_vac = (df_vac_raw
    .select('iso_code', 'country', F.explode('data').alias('d'))
    .select('iso_code', 'country',
            F.to_date('d.date').alias('date'),
            F.col('d.daily_vaccinations').cast('double').alias('daily_vaccinations'))
    .filter(~F.col('iso_code').startswith('OWID_'))
    .filter(F.col('daily_vaccinations').isNotNull()))

# Agrega por mes
df_monthly = (df_vac
    .withColumn('year', F.year('date'))
    .withColumn('month', F.month('date'))
    .groupBy('country', 'iso_code', 'year', 'month')
    .agg(F.sum('daily_vaccinations').alias('vaccinations_in_period')))

# Window pra rank
w_month = Window.partitionBy('year', 'month').orderBy(F.desc('vaccinations_in_period'))
df_top10_monthly = (df_monthly
    .withColumn('rank', F.row_number().over(w_month))
    .filter(F.col('rank') <= 10)
    .orderBy('year', 'month', 'rank'))

print('Q2.2 - Top 10 mensal (mostrando Jun/2021 e Jun/2022):')
df_top10_monthly.filter((F.col('year') == 2021) & (F.col('month') == 6)).show(truncate=False)
df_top10_monthly.filter((F.col('year') == 2022) & (F.col('month') == 6)).show(truncate=False)

Carregando vaccinations.json...
Q2.2 - Top 10 mensal (mostrando Jun/2021 e Jun/2022):
+-------------+--------+----+-----+----------------------+----+
|country      |iso_code|year|month|vaccinations_in_period|rank|
+-------------+--------+----+-----+----------------------+----+
|China        |CHN     |2021|6    |5.81455146E8          |1   |
|India        |IND     |2021|6    |1.13298089E8          |2   |
|Japan        |JPN     |2021|6    |3.4001095E7           |3   |
|Brazil       |BRA     |2021|6    |3.0533559E7           |4   |
|United States|USA     |2021|6    |2.5898051E7           |5   |
|Germany      |DEU     |2021|6    |2.4725217E7           |6   |
|Turkey       |TUR     |2021|6    |1.9298109E7           |7   |
|France       |FRA     |2021|6    |1.7398371E7           |8   |
|Italy        |ITA     |2021|6    |1.6582114E7           |9   |
|Mexico       |MEX     |2021|6    |1.5173302E7           |10  |
+-------------+--------+----+-----+----------------------+----+

+-------------+--

In [0]:
# Q2.3 - top 10 anual, ordenado por (n_vacinas DESC, vacinacoes DESC)
print('Q2.3 - Top 10 anual com vacinas usadas:')

# Total por pais por ano
df_yearly = (df_vac
    .withColumn('year', F.year('date'))
    .groupBy('country', 'iso_code', 'year')
    .agg(F.sum('daily_vaccinations').alias('vaccinations_in_year')))

# Join com numero de vacinas
df_locs_n = (df_loc_paises
    .select('iso_code',
            F.size(F.split(F.col('vaccines'), ',\\s*')).alias('n_vaccines')))

df_joined = df_yearly.join(df_locs_n, 'iso_code')

# Window: ORDENACAO COMBINADA (criterio do enunciado)
w_year = (Window.partitionBy('year')
    .orderBy(F.desc('n_vaccines'), F.desc('vaccinations_in_year')))

df_top10_yearly = (df_joined
    .withColumn('rank', F.row_number().over(w_year))
    .filter(F.col('rank') <= 10))

# Adiciona a lista de vacinas
df_locs_v = df_loc_paises.select('iso_code', F.col('vaccines').alias('vacinas_usadas'))
df_final = df_top10_yearly.join(df_locs_v, 'iso_code').orderBy('year', 'rank')

print('Resultado completo:')
df_final.select('year', 'rank', 'country', 'n_vaccines', 'vaccinations_in_year', 'vacinas_usadas') \
    .show(50, truncate=False)

Q2.3 - Top 10 anual com vacinas usadas:
Resultado completo:
+----+----+--------------------+----------+--------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|year|rank|country             |n_vaccines|vaccinations_in_year|vacinas_usadas                                                                                                                                                    |
+----+----+--------------------+----------+--------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|2020|1   |Bahrain             |10        |34657.0             |CanSino, Covaxin, Johnson&Johnson, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing, Sinovac, Sputnik Light, Sputnik V                             |
|2020|2   |Qatar            